In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
sns.set(style="ticks", context="notebook", palette="deep")
pd.set_option('display.max_columns', None)

In [3]:
path = "../../data/raw/"
dfs = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs[name].shape}")

Loaded 01_DiatomInventories_GTstudentproject_B with shape (1643872, 8)
Loaded 02_InfoSites_GTstudentproject_B with shape (8404, 11)
Loaded 03_IBD_GTstudentproject_test with shape (5063, 2)
Loaded 03_IBD_GTstudentproject_train with shape (43783, 4)
Loaded 04_PressureStatus_GTstudentproject_B with shape (49231, 30)
Loaded 05_EnvParamMeans_GTstudentproject_B with shape (3763903, 8)
Loaded 06_ListEnvParam_GNNprojectGT_B with shape (192, 14)
Loaded 07_TaxaCode_GTstudentproject_B with shape (2292, 2)


In [4]:
press = dfs[list(dfs.keys())[4]]
site = dfs[list(dfs.keys())[1]]


In [5]:
site

,CodeSite_SamplingOperations,Longitude_Lambert93,Latitude_Lambert93,Watershed,CodeDepartement,HERlvl1Code,HERlvl1Name,HERlvl2Code,HERlvl2Name,Altitude,Streamsize
0,S02096750,993887.00,6875414.00,Rhin-Meuse,67,10,COTES CALCAIRES EST,25,Plateau lorrain,225.0,None
1,S05170800,596306.00,6214010.00,Adour-Garonne,09,14,COTEAUX AQUITAINS,68,Coteaux molassiques Est Aquitaine,424.0,TP
2,S05201050,371740.00,6264090.00,Adour-Garonne,64,14,COTEAUX AQUITAINS,77,Coteaux molassiques bassin de l'adour,25.0,P
3,S02067400,984385.00,6837446.00,Rhin-Meuse,54,10,COTES CALCAIRES EST,25,Plateau lorrain,260.0,M
4,S05023100,488324.00,6553060.00,Adour-Garonne,16,9,TABLES CALCAIRES,97,TC - Charentes Poitou,99.0,TP
...,...,...,...,...,...,...,...,...,...,...,...
8399,S04150640,317312.08,6653804.73,Loire-Bretagne,85,12,ARMORICAIN,58,MA-sud interieur,0.0,TP
8400,S04431034,732377.00,6573686.00,Loire-Bretagne,03,17,DEPRESSIONS SEDIMENTAIRES,52,Fosses tectoniques,0.0,TP
8401,S06098890,846374.00,6495100.00,Rhône-Méditerranée,38,5,JURA-PREALPES DU NORD,85,Collines du Bas Dauphine,154.0,P
8402,S04447002,568526.60,6712332.00,Loire-Bretagne,41,9,TABLES CALCAIRES,54,TC-Nord Loire-Perche,0.0,M


In [6]:
her_stuff = pd.merge(press, site, on = 'CodeSite_SamplingOperations', how='left' )
her = her_stuff[['SamplingOperations_code','HERlvl1Code','HERlvl2Code']]

In [7]:
train =dfs[list(dfs.keys())[3]] # First parquet

In [8]:
train['IBD_x'] = train['IBD']/train['IBD_EQR']


In [9]:
train_sites = pd.merge(train, her, on='SamplingOperations_code', how = 'inner' )

In [10]:
train_sites

,SamplingOperations_code,IBD,IBD_EQR,IBD_EQR_Status,IBD_x,HERlvl1Code,HERlvl2Code
0,S04319000_20130724,5.2,0.256098,Bad,20.304762,12,118
1,S06172100_20080716,6.1,0.298246,Bad,20.452941,6,105
2,S03255920_20120802,3.7,0.157895,Bad,23.433333,9,37
3,S04064720_20150710,4.9,0.228070,Bad,21.484615,9,41
4,S05001500_20080726,5.3,0.251462,Bad,21.076744,9,97
...,...,...,...,...,...,...,...
43778,S06165700_20230628,6.8,0.339181,Poor,20.048276,6,56
43779,S06440770_20230524,15.3,0.836257,Good,18.295804,10,75
43780,S06710030_20230627,18.4,1.000000,High,18.400000,6,56
43781,S06036970_20230628,14.5,0.678571,Moderate,21.368421,15,81


In [11]:
train_sites[np.isinf(train_sites['IBD_x'])]


,SamplingOperations_code,IBD,IBD_EQR,IBD_EQR_Status,IBD_x,HERlvl1Code,HERlvl2Code
1135,S05224100_20080821,2.9,0.0,Bad,inf,13,21
2478,S02084800_20090811,1.0,0.0,Bad,inf,10,25
2524,S05224100_20090709,4.2,0.0,Bad,inf,13,21
4365,S05228000_20100808,4.6,0.0,Bad,inf,13,21
4493,S05224100_20100807,3.1,0.0,Bad,inf,13,21
5272,S06154000_20110725,4.8,0.0,Bad,inf,7,15
6857,S05224100_20120719,3.0,0.0,Bad,inf,13,21
10713,S04057770_20130627,5.0,0.0,Bad,inf,21,92
22901,S05053086_20170922,4.8,0.0,Bad,inf,21,92
29561,S05053086_20180917,4.4,0.0,Bad,inf,21,92


In [ ]:
train_sites_size = train_sites[train_sites[]]

In [12]:
tabla_resumen_lvl1 = (
    train_sites.groupby('HERlvl1Code')['IBD_x']
    .agg(
        valor_minimo='min',
        valor_maximo='max',
        media='mean',
        mediana='median',
        moda=lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        desviacion_estandar='std',
        varianza='var'
    )
    .reset_index()
)

tabla_resumen_lvl2 = (
    train_sites.groupby('HERlvl2Code')['IBD_x']
    .agg(
        valor_minimo='min',
        valor_maximo='max',
        media='mean',
        mediana='median',
        moda=lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        desviacion_estandar='std',
        varianza='var'
    )
    .reset_index()
)

In [13]:
tabla_resumen_lvl1

,HERlvl1Code,valor_minimo,valor_maximo,media,mediana,moda,desviacion_estandar,varianza
0,1,20.000000,37.058824,20.361099,20.000000,20.000000,1.024867,1.050352
1,2,20.000000,32.441860,20.275776,20.000000,20.000000,1.279045,1.635955
2,3,19.000000,247.333333,21.202753,20.194690,20.000000,5.616639,31.546638
3,4,19.000000,52.888889,21.234916,20.250000,20.000000,3.213041,10.323630
4,5,20.000000,77.500000,21.458913,20.905512,20.000000,2.759542,7.615071
5,6,18.100000,23.207143,18.715185,18.415385,20.000000,0.670439,0.449489
6,7,20.000000,inf,inf,20.000000,20.000000,NaN,NaN
7,8,19.000000,34.588235,20.088546,20.000000,20.000000,1.292652,1.670949
8,9,18.100000,23.433333,18.403138,18.321429,18.287500,0.314812,0.099107
9,10,18.100000,inf,inf,18.385714,20.000000,NaN,NaN


In [14]:
display(tabla_resumen_lvl2)

,HERlvl2Code,valor_minimo,valor_maximo,media,mediana,moda,desviacion_estandar,varianza
0,1,18.136364,20.000000,18.572428,18.304225,20.000000,0.612446,0.375090
1,2,20.000000,34.736842,21.190277,20.555556,20.000000,1.608304,2.586640
2,3,20.000000,23.333333,20.800885,20.514706,20.000000,0.851185,0.724517
3,4,19.263158,23.722222,20.233513,20.000000,20.000000,0.950349,0.903164
4,5,20.000000,49.090909,21.022015,20.102041,20.000000,2.977841,8.867535
...,...,...,...,...,...,...,...,...
101,115,18.200000,20.000000,19.676471,20.000000,20.000000,0.552867,0.305662
102,116,18.203226,20.000000,18.982056,18.862500,18.203226,0.795652,0.633062
103,117,17.406135,20.213953,17.865103,17.789831,17.789831,0.385594,0.148683
104,118,17.406135,21.085714,17.801190,17.712000,17.744262,0.379449,0.143982
